# NEXUS — RL Fine-Tuning with Verifiable Rewards (RLVR)

This notebook fine-tunes a small open code model so it **learns from the same verifier NEXUS uses in production** — the reward signal isn't a learned reward model, it's a direct pass/fail from formal + property-based checks (the same technique behind DeepSeek-R1 and most 2025–2026 reasoning-model post-training).

**Runtime:** Free Colab T4 GPU is enough (`Runtime -> Change runtime type -> T4 GPU`).
**Base model:** `Qwen/Qwen2.5-Coder-0.5B-Instruct` — small enough to fully fine-tune on a free T4, strong enough to show a real before/after signal.
**Time:** ~20-40 minutes end to end on a free T4.

**Honesty note:** this notebook prints the actual numbers it gets when you run it — there are no pre-filled "expected results" anywhere here. Whatever pass@1 improvement (or lack of one) you see is the real result on your run. Put those real numbers in your README/resume, not invented ones.

## 1. Install dependencies

In [ ]:
!pip install -q -U transformers==4.47.1 trl==0.13.0 peft==0.14.0 accelerate==1.2.1 datasets==3.2.0 bitsandbytes==0.45.0

## 2. Minimal self-contained problem bank + verifier
Same logic as `backend/app/core/verifier.py` and `problems.py`, trimmed to two problem types so the notebook is fully standalone. If you have the repo cloned, you can `sys.path.append('nexus/backend')` and import the real modules instead.

In [ ]:
import random, re, signal, contextlib, io

class TimeoutException(Exception):
    pass

@contextlib.contextmanager
def time_limit(seconds):
    def handler(signum, frame):
        raise TimeoutException()
    old = signal.signal(signal.SIGALRM, handler)
    signal.alarm(seconds)
    try:
        yield
    finally:
        signal.alarm(0)
        signal.signal(signal.SIGALRM, old)

SAFE_BUILTINS = {k: __builtins__[k] if isinstance(__builtins__, dict) else getattr(__builtins__, k)
                 for k in ['range','len','sorted','list','dict','set','tuple','str','int','float',
                           'bool','min','max','sum','enumerate','zip','map','filter','abs','round',
                           'isinstance','reversed','all','any']}

def extract_code(text):
    m = re.search(r'```(?:python)?\s*([\s\S]*?)```', text)
    return (m.group(1) if m else text).strip()

def safe_run(code, entry_point, args, timeout=2):
    ns = {'__builtins__': SAFE_BUILTINS}
    try:
        with time_limit(timeout):
            exec(compile(code, '<candidate>', 'exec'), ns)
            fn = ns.get(entry_point)
            if fn is None:
                return None, f"NameError: '{entry_point}' not defined"
            return fn(*args), None
    except TimeoutException:
        return None, 'Timeout'
    except Exception as e:
        return None, f'{type(e).__name__}: {e}'

def verify_sort(arr, out):
    if not isinstance(out, list) or len(out) != len(arr):
        return False
    if sorted(out) != sorted(arr):
        return False
    return all(out[i] <= out[i+1] for i in range(len(out)-1))

def verify_two_sum(arr, target, out):
    if out == []:
        return not any(arr[a]+arr[b]==target for a in range(len(arr)) for b in range(len(arr)) if a != b)
    if not (isinstance(out, list) and len(out) == 2):
        return False
    i, j = out
    if not (isinstance(i, int) and isinstance(j, int) and 0 <= i < len(arr) and 0 <= j < len(arr) and i != j):
        return False
    return arr[i] + arr[j] == target

PROBLEMS = {
    'sort_list': {
        'prompt': 'Write a Python function `sort_list(arr)` that returns a NEW list with the integers in arr sorted in non-decreasing order. Implement the sort yourself, do not use sorted() or .sort(). Return ONLY the function in a python code block.',
        'entry_point': 'sort_list',
        'gen_input': lambda rng: [[rng.randint(-50, 50) for _ in range(rng.randint(0, 12))]],
        'verify': lambda inp, out: verify_sort(inp[0], out),
    },
    'two_sum': {
        'prompt': 'Write a Python function `two_sum(arr, target)` that returns a list [i, j] of two distinct indices such that arr[i]+arr[j]==target, or [] if no such pair exists. Return ONLY the function in a python code block.',
        'entry_point': 'two_sum',
        'gen_input': lambda rng: (lambda n: [[rng.randint(-30,30) for _ in range(n)], rng.randint(-60,60)])(rng.randint(2, 10)),
        'verify': lambda inp, out: verify_two_sum(inp[0], inp[1], out),
    },
}
print('Problem bank ready:', list(PROBLEMS.keys()))

## 3. Build the training dataset (prompts only — rewards are computed live during GRPO rollout)

In [ ]:
from datasets import Dataset

rng = random.Random(0)
rows = []
for _ in range(300):
    pid = rng.choice(list(PROBLEMS.keys()))
    rows.append({'prompt': PROBLEMS[pid]['prompt'], 'problem_id': pid})

train_dataset = Dataset.from_list(rows)
print(train_dataset)
print(train_dataset[0])

## 4. Reward function — this IS the RLVR signal
For each generated completion, we run it through the same kind of sandboxed exec + verifier check NEXUS uses in production, on a handful of fresh random test cases, and return the pass rate as the reward.

In [ ]:
def reward_fn(completions, problem_id, **kwargs):
    rewards = []
    eval_rng = random.Random(123)
    for completion, pid in zip(completions, problem_id):
        spec = PROBLEMS[pid]
        code = extract_code(completion)
        passed, total = 0, 5
        for _ in range(total):
            args = spec['gen_input'](eval_rng)
            out, err = safe_run(code, spec['entry_point'], args)
            if err is None and spec['verify'](args, out):
                passed += 1
        rewards.append(passed / total)
    return rewards

# sanity check the reward function on a hand-written correct + buggy solution
good = '```python\ndef sort_list(arr):\n    a=list(arr)\n    for i in range(len(a)):\n        for j in range(len(a)-i-1):\n            if a[j]>a[j+1]: a[j],a[j+1]=a[j+1],a[j]\n    return a\n```'
bad = '```python\ndef sort_list(arr):\n    return arr\n```'
print('good solution reward:', reward_fn([good], ['sort_list']))
print('bad solution reward:', reward_fn([bad], ['sort_list']))

## 5. Load base model + LoRA config

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig

MODEL_NAME = 'Qwen/Qwen2.5-Coder-0.5B-Instruct'

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.bfloat16, device_map='auto'
)

lora_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
    task_type='CAUSAL_LM',
)
print('Model + LoRA config ready.')

## 6. Baseline evaluation (BEFORE training) — real numbers, run live

In [ ]:
def evaluate_pass_at_1(model, tokenizer, n_problems=30, seed=999):
    eval_rng = random.Random(seed)
    correct = 0
    per_problem = {}
    for _ in range(n_problems):
        pid = eval_rng.choice(list(PROBLEMS.keys()))
        spec = PROBLEMS[pid]
        messages = [{'role': 'user', 'content': spec['prompt']}]
        inputs = tokenizer.apply_chat_template(messages, return_tensors='pt', add_generation_prompt=True).to(model.device)
        with torch.no_grad():
            out_ids = model.generate(inputs, max_new_tokens=300, do_sample=False, pad_token_id=tokenizer.eos_token_id)
        text = tokenizer.decode(out_ids[0][inputs.shape[1]:], skip_special_tokens=True)
        code = extract_code(text)
        ok_count = 0
        for _ in range(5):
            args = spec['gen_input'](eval_rng)
            result, err = safe_run(code, spec['entry_point'], args)
            if err is None and spec['verify'](args, result):
                ok_count += 1
        is_correct = ok_count == 5  # strict pass@1: must pass ALL sampled cases
        correct += int(is_correct)
        per_problem[pid] = per_problem.get(pid, [0, 0])
        per_problem[pid][0] += int(is_correct)
        per_problem[pid][1] += 1
    return correct / n_problems, per_problem

baseline_score, baseline_breakdown = evaluate_pass_at_1(model, tokenizer)
print(f'BASELINE pass@1 (strict, all-test-cases): {baseline_score:.1%}')
print('Breakdown:', baseline_breakdown)

## 7. GRPO training (Reinforcement Learning with Verifiable Rewards)

In [ ]:
from trl import GRPOConfig, GRPOTrainer

training_args = GRPOConfig(
    output_dir='nexus-grpo-checkpoint',
    learning_rate=1e-5,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    num_generations=4,          # rollouts sampled per prompt, needed for GRPO's group advantage
    max_prompt_length=256,
    max_completion_length=320,
    num_train_epochs=1,
    logging_steps=5,
    save_strategy='no',
    bf16=True,
    report_to=[],
)

trainer = GRPOTrainer(
    model=model,
    reward_funcs=reward_fn,
    args=training_args,
    train_dataset=train_dataset,
    peft_config=lora_config,
)

trainer.train()

## 8. Evaluation AFTER training — compare honestly against the baseline above

In [ ]:
trained_score, trained_breakdown = evaluate_pass_at_1(trainer.model, tokenizer)
print(f'BASELINE pass@1: {baseline_score:.1%}')
print(f'AFTER GRPO pass@1: {trained_score:.1%}')
delta = (trained_score - baseline_score) * 100
print(f'Delta: {delta:+.1f} percentage points')
print()
print('Per-problem breakdown (correct/total):')
print(' Baseline:', baseline_breakdown)
print(' After:   ', trained_breakdown)
print()
print('>>> Copy these REAL numbers into your README / resume bullet. Do not invent different ones. <<<')

## 9. Save the LoRA adapter (optional — for loading into the backend later)

In [ ]:
trainer.model.save_pretrained('nexus-lora-adapter')
tokenizer.save_pretrained('nexus-lora-adapter')

import shutil
shutil.make_archive('nexus-lora-adapter', 'zip', 'nexus-lora-adapter')
print('Saved and zipped: nexus-lora-adapter.zip — download it from the Colab file browser on the left.')

## Troubleshooting
- **`trl`/`GRPOConfig` import errors**: TRL's GRPO API moved fast through 2025; if signatures changed, check `pip show trl` and the GRPOTrainer docs for your installed version — the core idea (reward_funcs + GRPOConfig + GRPOTrainer) is stable even if exact kwarg names shift.
- **Out of memory on T4**: lower `per_device_train_batch_size` to 2 and `num_generations` to 2.
- **Reward stays at 0 the whole time**: print a few raw `completions` inside `reward_fn` to check the model is actually emitting parseable ```python blocks — small base models sometimes need a one-shot example added to the prompt.